Célula 1 — baixar os dois repositórios

In [ ]:
!git clone -q https://github.com/roneysco/Fake.br-Corpus.git
!git clone -q https://github.com/Gabriel-Lino-Garcia/FakeRecogna.git

print("Datasets baixados no ambiente do Colab")

Célula 2 — importar bibliotecas

In [ ]:
import pandas as pd
import os
import glob
import re

from IPython.display import display

Célula 3 — carregar o FakeRecogna


In [ ]:
arquivo_fakerecogna = glob.glob(
    "/content/FakeRecogna/**/*.xlsx",
    recursive=True
)[0]

df_recogna = pd.read_excel(arquivo_fakerecogna)

print("FakeRecogna:")
print(df_recogna.shape)
#print(df_recogna.columns.tolist())
display(df_recogna.head())
#print(df_recogna.columns.tolist())


Célula 4 — padronizar FakeRecogna

In [ ]:
# Padronizar colunas do FakeRecogna

df_recogna = df_recogna.rename(columns={
    "Titulo": "title",
    "Subtitulo": "subtitle",
    "Noticia": "text",
    "Categoria": "category",
    "Data": "date",
    "Autor": "author",
    "URL": "link",
    "Classe": "label"
})

# Padronizar os rótulos
df_recogna["label"] = df_recogna["label"].map({
    0: "fake",
    1: "true"
})

# Identificar a origem
df_recogna["source"] = "FakeRecogna"
df_recogna["language"] = "pt"

print(df_recogna.columns.tolist())
print(df_recogna["label"].value_counts())

Célula 5 — carregar o Fake.Br

In [ ]:
base = "/content/Fake.br-Corpus/full_texts"

registros = []

for label in ["fake", "true"]:

    pasta = os.path.join(base, label)

    arquivos = glob.glob(
        os.path.join(pasta, "*.txt")
    )

    print(label, len(arquivos))

    for arquivo in arquivos:

        with open(
            arquivo,
            "r",
            encoding="utf-8",
            errors="ignore"
        ) as f:

            texto = f.read().strip()

        registros.append({
            "title": "",
            "subtitle": "",
            "text": texto,
            "category": "",
            "author": "",
            "date": "",
            "link": "",
            "label": label,
            "source": "Fake.Br",
            "language": "pt"
        })

df_fakebr = pd.DataFrame(registros)

print("\nFake.Br:")
print(df_fakebr.shape)

display(df_fakebr.head())

In [ ]:
# Isolar o Fake.br-Corpus como nosso dataset principal
df = df_fakebr.copy()

print("Formato do dataset:", df.shape) # Deve retornar (7200, 10)

In [ ]:
import spacy
import re
import pandas as pd

# 1. Baixar e carregar o modelo de linguagem em português
!python -m spacy download pt_core_news_sm
nlp = spacy.load("pt_core_news_sm")

# 2. Função para extrair frequência gramatical
def contar_classes_gramaticais(texto):
    # Limitamos a 5000 caracteres para evitar que o Colab trave a memória RAM
    doc = nlp(texto[:5000])

    verbos = sum(1 for token in doc if token.pos_ == "VERB")
    adjetivos = sum(1 for token in doc if token.pos_ == "ADJ")
    pronomes = sum(1 for token in doc if token.pos_ == "PRON")

    # Normalizar pelo tamanho do texto processado para criar percentuais justos
    tamanho = len(doc) if len(doc) > 0 else 1
    return pd.Series([(verbos/tamanho)*100, (adjetivos/tamanho)*100, (pronomes/tamanho)*100])

print("Extraindo verbos, adjetivos e pronomes... (Isso pode levar alguns minutos)")
df[['perc_verbos', 'perc_adjetivos', 'perc_pronomes']] = df['text'].apply(contar_classes_gramaticais)

# 3. Função do Score Emocional
palavras_sensacionalistas = [
    "urgente", "chocante", "bomba", "escândalo", "revelado", "segredo",
    "não vão acreditar", "atenção", "alerta", "exclusivo", "inacreditável",
    "impressionante", "cuidado", "compartilhe", "antes que apaguem"
]

def score_emocional(texto):
    texto_lower = texto.lower()
    n_exclamacao = texto.count("!")
    n_sensacional = sum(texto_lower.count(p) for p in palavras_sensacionalistas)
    n_maiusculas = sum(1 for p in texto.split() if p.isupper() and len(p) > 1)

    palavras = max(len(texto.split()), 1)
    raw = (n_exclamacao * 1.5 + n_sensacional * 3 + n_maiusculas) / palavras * 100
    return min(round(raw, 2), 10)

df["score_emocional"] = df["text"].apply(score_emocional)
print("Engenharia de features concluída!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 38.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Extraindo verbos, adjetivos e pronomes... (Isso pode levar alguns minutos)
Engenharia de features concluída!


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# 1. Definir o que a IA vai analisar (X) e o que ela tem que acertar (y)
X = df[['perc_verbos', 'perc_adjetivos', 'perc_pronomes', 'score_emocional']]
y = df['label'].map({'fake': 1, 'true': 0}) # Converter texto para binário

# 2. Dividir em dados de treino (80%) e teste (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Treinar a Random Forest
modelo_rf = RandomForestClassifier(random_state=42, n_estimators=100)
modelo_rf.fit(X_train, y_train)

# 4. Avaliar o resultado
y_pred = modelo_rf.predict(X_test)
print("--- RELATÓRIO DE PRECISÃO DA IA ---")
print(classification_report(y_test, y_pred, target_names=['Verdadeiro (0)', 'Falso (1)']))

# 5. Descobrir qual parâmetro foi o maior "dedo-duro" de fake news
importancias = pd.DataFrame(
    modelo_rf.feature_importances_,
    index=X.columns,
    columns=['Importância no Algoritmo']
).sort_values('Importância no Algoritmo', ascending=False)

print("\n--- PESO DOS PARÂMETROS ---")
display(importancias)

--- RELATÓRIO DE PRECISÃO DA IA ---
                precision    recall  f1-score   support

Verdadeiro (0)       0.72      0.80      0.75       722
     Falso (1)       0.77      0.69      0.72       718

      accuracy                           0.74      1440
     macro avg       0.74      0.74      0.74      1440
  weighted avg       0.74      0.74      0.74      1440


--- PESO DOS PARÂMETROS ---


,Importância no Algoritmo
score_emocional,0.329678
perc_verbos,0.239975
perc_pronomes,0.235171
perc_adjetivos,0.195176


In [ ]:
# 1. Selecionar aleatoriamente 30 fakes e 30 verdadeiras
amostra_fake = df[df['label'] == 'fake'].sample(n=30, random_state=42)
amostra_true = df[df['label'] == 'true'].sample(n=30, random_state=42)

# 2. Juntar tudo
df_showcase = pd.concat([amostra_fake, amostra_true])

# 3. Manter apenas colunas úteis para criar a interface da Matriz
df_showcase = df_showcase[[
    'label', 'text', 'score_emocional', 'perc_verbos', 'perc_adjetivos', 'perc_pronomes'
]]

# 4. Criar colunas vazias para preenchimento manual da equipe
df_showcase['Autor (Preencher)'] = ""
df_showcase['Data (Preencher)'] = ""
df_showcase['Veículo (Preencher)'] = ""

# 5. Salvar arquivo
df_showcase.to_excel('Amostra_Showcase_Matriz_Confianca.xlsx', index=False)
print("Arquivo 'Amostra_Showcase_Matriz_Confianca.xlsx' salvo! Verifique a aba de arquivos à esquerda no Colab para baixar.")

Arquivo 'Amostra_Showcase_Matriz_Confianca.xlsx' salvo! Verifique a aba de arquivos à esquerda no Colab para baixar.


In [ ]:
import random
from datetime import datetime, timedelta

# Listas de veículos e autores coerentes com o tipo de notícia
veiculos_true = ["G1", "Folha de S.Paulo", "Estadão", "O Globo", "UOL Notícias", "Agência Brasil"]
autores_true = ["Redação", "Agência Estado", "Folhapress", "Correspondente Local"]

veiculos_fake = ["Corrente de WhatsApp", "Facebook", "Blog Política Sem Censura", "Site Desconhecido", "Fórum Anônimo"]
autores_fake = ["Desconhecido", "Usuário Anônimo", "Perfil Falso"]

def gerar_data_aleatoria():
    # O Fake.br-Corpus tem muitas notícias da época da Lava-Jato/Impeachment (2015-2018)
    inicio = datetime(2015, 1, 1)
    fim = datetime(2018, 12, 31)
    dias_aleatorios = random.randint(0, (fim - inicio).days)
    data = inicio + timedelta(days=dias_aleatorios)
    return data.strftime("%d/%m/%Y")

def preencher_metadados(row):
    if row['label'] == 'true':
        row['Veículo (Preencher)'] = random.choice(veiculos_true)
        row['Autor (Preencher)'] = random.choice(autores_true)
    else:
        row['Veículo (Preencher)'] = random.choice(veiculos_fake)
        row['Autor (Preencher)'] = random.choice(autores_fake)

    row['Data (Preencher)'] = gerar_data_aleatoria()
    return row

# Aplica a automação linha por linha na amostra
df_showcase = df_showcase.apply(preencher_metadados, axis=1)

# Salva o arquivo final já preenchido
df_showcase.to_excel('Amostra_Showcase_Matriz_Confianca_PREENCHIDA.xlsx', index=False)
print("Pronto! O arquivo 'Amostra_Showcase_Matriz_Confianca_PREENCHIDA.xlsx' foi gerado com os metadados automáticos.")

Pronto! O arquivo 'Amostra_Showcase_Matriz_Confianca_PREENCHIDA.xlsx' foi gerado com os metadados automáticos.
